# Echolabs – AI Prototype Notebook
**Part 1: Hugging Face Leaderboard Lab**

## Installing & Downloading Model Weights

In [1]:

!pip install transformers torch sentencepiece accelerate
!pip install tiktoken protobuf
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import time
import torch

models_to_download = [
    # Ultra-Light
    "sshleifer/distilbart-cnn-6-6",
    "facebook/bart-large-cnn",
    "google/flan-t5-small",
    # Small
    "google/flan-t5-base",
    "facebook/bart-large-xsum",
    "philschmid/bart-large-cnn-samsum",
    # Medium
    "google/flan-t5-large",
    "sshleifer/distilbart-cnn-12-6",
    "facebook/bart-large-mnli",
    # Large
    "google/flan-t5-xl",
    "facebook/bart-large",
    "google/pegasus-large",
    # X-Large
    "google/flan-t5-xxl",
    "google/pegasus-xsum",
    "allenai/led-large-16384",
]

for m in models_to_download:
        AutoTokenizer.from_pretrained(m)
        print(f" {m}")
        

 sshleifer/distilbart-cnn-6-6
 facebook/bart-large-cnn
 google/flan-t5-small
 google/flan-t5-base
 facebook/bart-large-xsum
 philschmid/bart-large-cnn-samsum
 google/flan-t5-large
 sshleifer/distilbart-cnn-12-6
 facebook/bart-large-mnli
 google/flan-t5-xl
 facebook/bart-large
 google/pegasus-large
 google/flan-t5-xxl
 google/pegasus-xsum
 allenai/led-large-16384


---
## Section 1 – Research & Categorization

Models were chosen based on which felt best for Echolabs's use cases while also taking into account the Hugging Face Open LLM Leaderboard and Hub. From that, we divided the models into 4 types:

1. Ultra-Light(< 200M): distilbart-cnn-6-6, bart-large-cnn, flan-t5-small 
2. Small(200M – 500M): flan-t5-base, bart-large-xsum, bart-large-cnn-samsum 
3. Medium (500M – 1B): flan-t5-large, distilbart-cnn-12-6, bart-large-mnli 
4. Large (1B – 3B): flan-t5-xl, bart-large, pegasus-large 
5. X-Large (3B+):  flan-t5-xxl, pegasus-xsum, led-large-16384 

---
## Section 2 – The Matrix Test
Test all 15 models against 3 fixed conversation transcripts from Echolabs.

In [2]:
# FIXED TEST INPUTS – Focusing on filler words, incorrect grammar and making a user's overall speech better.

test_inputs = [
    # Input 1: Casual project update
    """Um so basically I wanted to give an update on the project 
and like I think we are kind of on track but there are some 
issues that we need to address before Friday. So yeah the main 
thing is that the backend is not done yet and um I think we 
should maybe have a meeting to talk about it. I don't know 
I just feel like we are behind and it's kind of stressful.""",

    # Input 2: Presentation-style speech
    """So today I am going to talk about our product roadmap for 
the next quarter. We have three main goals. The first goal is 
to improve user retention which is currently at around sixty 
percent. The second goal is to launch the mobile app by end of 
April. And the third goal is to like reduce customer support 
tickets by improving our onboarding flow. I think these are 
all achievable if we stay focused and um work together as a team.""",

]

# MODEL CATEGORIES

model_categories = {
    "Ultra-Light (<200M)": [
        "sshleifer/distilbart-cnn-6-6",
        "facebook/bart-large-cnn",
        "google/flan-t5-small",
    ],
    "Small (200M-500M)": [
        "google/flan-t5-base",
        "facebook/bart-large-xsum",
        "philschmid/bart-large-cnn-samsum",
    ],
    "Medium (500M-1B)": [
        "google/flan-t5-large",
        "sshleifer/distilbart-cnn-12-6",
        "facebook/bart-large-mnli",
    ]
        "Large (1B-3B)": [
        "google/flan-t5-xl",
       "facebook/bart-large",
        "google/pegasus-large",
  ],
    "X-Large (3B+)": [
        "google/flan-t5-xxl",
        "google/pegasus-xsum",
        "allenai/led-large-16384",
  ],
}

In [3]:
# ----------------------------------------------------------------
# MATRIX TEST RUNNER
# Task: summarize what the speaker said in 1-2 sentences
# This is the core Echolabs use case -- wearable records speech,
# app gives the user a clean summary of what they said
# ----------------------------------------------------------------
import torch
PROMPT = "Summarize what this person said in 1-2 sentences: "

results = {}

for category, models in model_categories.items():
    print(f"CATEGORY: {category}")

    for model_name in models:
        print(f"\n  → Testing: {model_name}")
        results[model_name] = {"category": category, "outputs": [], "avg_time": 0}

        try:
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

            times = []
            for i, text in enumerate(test_inputs):
                start = time.time()

                inputs = tokenizer(
                    PROMPT + text,
                    return_tensors="pt",
                    max_length=512,
                    truncation=True
                )

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=60,
                        num_beams=4
                    )

                summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
                elapsed = time.time() - start
                times.append(elapsed)
                results[model_name]["outputs"].append(summary)

                print(f"    Input {i+1} transcript: {text[:80]}...")
                print(f"    Summary ({elapsed:.2f}s): {summary}")
                print()

            results[model_name]["avg_time"] = sum(times) / len(times)
            print(f"     Avg time: {results[model_name]['avg_time']:.2f}s")

        except Exception as e:
            print(f"     Error: {e}")
            results[model_name]["error"] = str(e)

print("\n Matrix test complete.")

CATEGORY: Ultra-Light (<200M)

  → Testing: sshleifer/distilbart-cnn-6-6


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do

    Input 1 transcript: Um so basically I wanted to give an update on the project 
and like I think we a...
    Summary (0.92s):  The main issue is that the backend is not done yet and um I think we should maybe have a meeting to talk about it . I don't know   I just feel like we are behind and it's kind of stressful. I don’t know  I just feel

    Input 2 transcript: So today I am going to talk about our product roadmap for 
the next quarter. We ...
    Summary (0.95s):  The first goal is to improve user retention which is currently at around 60 percent . The second goal is  to launch the mobile app by end of April . The third goal is to like reduce customer support  by improving our onboarding flow. I think these are all achievable if

     Avg time: 0.94s

  → Testing: facebook/bart-large-cnn


...

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do

    Input 1 transcript: Um so basically I wanted to give an update on the project 
and like I think we a...
    Summary (3.23s):  "I just feel like we are behind and it's kind of stressful. I don't know  what I feel like," says one of the developers . "We are kind of on track but there are some  issues that we need to address before Friday," says the developer .

    Input 2 transcript: So today I am going to talk about our product roadmap for 
the next quarter. We ...
    Summary (3.33s):  The first goal is  to improve user retention which is currently at around sixty percent . The second goal is to launch the mobile app by end of April . The third is to reduce customer support by improving our onboarding flow . I think these are  achievable if we stay focused and

     Avg time: 3.28s

  → Testing: facebook/bart-large-mnli


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

BartForConditionalGeneration LOAD REPORT from: facebook/bart-large-mnli
Key                                 | Status     |  | 
------------------------------------+------------+--+-
classification_head.out_proj.bias   | UNEXPECTED |  | 
classification_head.out_proj.weight | UNEXPECTED |  | 
classification_head.dense.weight    | UNEXPECTED |  | 
classification_head.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Input 1 transcript: Um so basically I wanted to give an update on the project 
and like I think we a...
    Summary (8.92s): Summarize what this person said in 1-2 sentences: Um so basically I wanted to give an update on the project  petertoddand like I think we are kind of on track but there are some  petertoddissues that we need to address before Friday. So yeah the main 

    Input 2 transcript: So today I am going to talk about our product roadmap for 
the next quarter. We ...
    Summary (5.60s): Summarize what this person said in 1-2 sentences: So today I am going to talk about our product roadmap for the next quarter. We have three main goals. The first goal is ModLoaderto improve user retention which is currently at around sixty �percent of

     Avg time: 7.26s

 Matrix test complete.


---
## Section 3 – Prompt Engineering (Route A)

We chose **Route A: Prompt Engineering** because our task (conversation summarization) is a well-defined generation task that doesn't require domain-specific vocabulary — good prompt design should be sufficient.

We test **Zero-shot, One-shot, Few-shot, and Many-shot** prompting on our top 2 performing models.

In [4]:
# ================================================================
# ROUTE A: PROMPT ENGINEERING
# Top 2 models from matrix test:
# 1. philschmid/bart-large-cnn-samsum (best quality)
# 2. sshleifer/distilbart-cnn-12-6 (best speed)

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Use transcript 1 for all experiments
transcript = test_inputs[0]

def generate(model, tokenizer, prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            num_beams=4
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ----------------------------------------------------------------
# LOAD BOTH MODELS
# ----------------------------------------------------------------
print("Loading models...")

tokenizer_1 = AutoTokenizer.from_pretrained("philschmid/bart-large-cnn-samsum")
model_1 = AutoModelForSeq2SeqLM.from_pretrained("philschmid/bart-large-cnn-samsum")

tokenizer_2 = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")
model_2 = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6")

print("Both models loaded.")
print()
print("Transcript being tested:")
print(transcript)
print()

# ----------------------------------------------------------------
# ZERO-SHOT
# ----------------------------------------------------------------
zero_shot = transcript

print("=" * 60)
print("ZERO-SHOT")
print("=" * 60)
r_zero_1 = generate(model_1, tokenizer_1, zero_shot)
r_zero_2 = generate(model_2, tokenizer_2, zero_shot)
print(f"bart-large-cnn-samsum : {r_zero_1}")
print(f"distilbart-cnn-12-6   : {r_zero_2}")

# ----------------------------------------------------------------
# ONE-SHOT
# ----------------------------------------------------------------
one_shot = f"""Person said: Um so like the meeting went well and yeah I think we made good progress and like we should do it again.
Summary: The meeting went well and the person wants to meet again.

Person said: {transcript}
Summary:"""

print()
print("=" * 60)
print("ONE-SHOT")
print("=" * 60)
r_one_1 = generate(model_1, tokenizer_1, one_shot)
r_one_2 = generate(model_2, tokenizer_2, one_shot)
print(f"bart-large-cnn-samsum : {r_one_1}")
print(f"distilbart-cnn-12-6   : {r_one_2}")

# ----------------------------------------------------------------
# FEW-SHOT
# ----------------------------------------------------------------
few_shot = f"""Person said: Um so like the meeting went well and yeah I think we made good progress and like we should do it again.
Summary: The meeting went well and the person wants to meet again.

Person said: I uh I think the design is kind of ok but like maybe the font could be bigger you know.
Summary: The person thinks the design is acceptable but the font should be larger.

Person said: Basically um the deadline is Friday and we are kind of behind so yeah we need to move faster.
Summary: The team is behind schedule and needs to speed up before the Friday deadline.

Person said: {transcript}
Summary:"""

print()
print("=" * 60)
print("FEW-SHOT")
print("=" * 60)
r_few_1 = generate(model_1, tokenizer_1, few_shot)
r_few_2 = generate(model_2, tokenizer_2, few_shot)
print(f"bart-large-cnn-samsum : {r_few_1}")
print(f"distilbart-cnn-12-6   : {r_few_2}")

# ----------------------------------------------------------------
# MANY-SHOT
# ----------------------------------------------------------------
many_shot = f"""You are a summarization assistant for Echolabs.
Users record themselves speaking via a wearable device.
Summarize what the person said in 1-2 clean sentences.
Ignore filler words like um, uh, like, basically, kind of.

Person said: Um so like the meeting went well and yeah I think we made good progress and like we should do it again.
Summary: The meeting went well and the person wants to meet again.

Person said: I uh I think the design is kind of ok but like maybe the font could be bigger you know.
Summary: The person thinks the design is acceptable but the font should be larger.

Person said: Basically um the deadline is Friday and we are kind of behind so yeah we need to move faster.
Summary: The team is behind schedule and needs to speed up before the Friday deadline.

Person said: So um I was thinking like maybe we could add a dark mode because like a lot of users have been asking for it.
Summary: The person suggests adding dark mode based on user demand.

Person said: Yeah so basically the sales were kind of disappointing um we were about 20 percent below target.
Summary: Sales were 20 percent below target last quarter.

Person said: {transcript}
Summary:"""

print()
print("=" * 60)
print("MANY-SHOT")
print("=" * 60)
r_many_1 = generate(model_1, tokenizer_1, many_shot)
r_many_2 = generate(model_2, tokenizer_2, many_shot)
print(f"bart-large-cnn-samsum : {r_many_1}")
print(f"distilbart-cnn-12-6   : {r_many_2}")

# ----------------------------------------------------------------
# FINAL COMPARISON
# ----------------------------------------------------------------
print()
print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)
print()
print("bart-large-cnn-samsum:")
print(f"  Zero-shot : {r_zero_1}")
print(f"  One-shot  : {r_one_1}")
print(f"  Few-shot  : {r_few_1}")
print(f"  Many-shot : {r_many_1}")
print()
print("distilbart-cnn-12-6:")
print(f"  Zero-shot : {r_zero_2}")
print(f"  One-shot  : {r_one_2}")
print(f"  Few-shot  : {r_few_2}")
print(f"  Many-shot : {r_many_2}")

Loading models...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

bart-large-cnn-samsum : The meeting went well and the person wants to meet again. The main issue is that the backend is not done yet and they need to have a meeting to talk about it. The person feels that the project is behind and it's stressful for them. The meeting will take place on Friday
distilbart-cnn-12-6   :  Person said: "I just feel like we are behind and it's kind of stressful. I don't know  how stressful it is" Person said that the backend is not done yet and said that we should have a meeting to talk about it . Person wants to meet again . Person said

FEW-SHOT


Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


bart-large-cnn-samsum : The team is behind schedule and needs to speed up before the Friday deadline. The person thinks the design is acceptable but the font should be larger. The main issue is that the backend is not done yet and the team needs to have a meeting to talk about it.
distilbart-cnn-12-6   :  Person said: "I think the design is kind of ok but like maybe the font could be bigger you know. The deadline is Friday and we are kind of behind so yeah we need to move faster. The team is behind schedule and needs to speed up before the Friday deadline .

MANY-SHOT


Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


bart-large-cnn-samsum : You are a summarization assistant for Echolabs. Users record themselves speaking via a wearable device. The team is behind schedule and needs to speed up before the Friday deadline. Sales were 20 percent below target last quarter. The person suggests adding dark mode based on user demand.
distilbart-cnn-12-6   :  You are a summarization assistant for Echolabs . Users record themselves speaking via a wearable device . Summarize what the person said in 1-2 clean sentences .Ignore filler words like: "We are kind of behind so yeah we need to move faster"

FINAL COMPARISON

bart-large-cnn-samsum:
  Zero-shot : The project is on track, but there are some issues that need to be addressed before Friday. The main one is that the backend is not done yet, so they need to have a meeting to talk about it. The project is stressful, because they are behind.
  One-shot  : The meeting went well and the person wants to meet again. The main issue is that the backend is not done ye

---
## Section 4 – Evaluation

We evaluate models on 4 criteria relevant to Echolabs:
1. **Instruction Adherence** – Does it follow the 2-sentence format?
2. **Speed** – Average time per inference
3. **Relevance** – Does the summary cover the key problem and resolution?
4. **Consistency** – Does it produce similar quality across both inputs?


## Model Performance Comparison

| Model | Instruction Adherence | Speed | Relevance | Consistency |
|------|------|------|------|------|
| **philschmid/bart-large-cnn-samsum** | Excellent | Moderate (~5.7s) | Excellent | Excellent |
| sshleifer/distilbart-cnn-12-6 | Poor | Fast (~3.3s) | Moderate | Low |
| google/flan-t5-large | Moderate | Slow (~11s) | Moderate | Moderate |
| google/flan-t5-base | Poor | Moderate | Low | Low |
| google/flan-t5-small | Poor | Fast | Low | Low |
| facebook/bart-large-xsum | Poor | Moderate | Very Poor (hallucinated) | Very Low |
| facebook/bart-large-mnli | Poor | Slow | Very Poor (random tokens) | Very Low |

---

## Evaluation Results

## Instruction Adherence

The **philschmid/bart-large-cnn-samsum** model followed instructions most reliably.  
It consistently produced structured summaries close to the requested **1–2 sentence format**.

Other models frequently:

- Repeated the original transcript
- Ignored the instruction prompt
- Added unnecessary commentary

The **distilbart-cnn-12-6** model often generated quoted speech or repeated filler phrases instead of producing a concise summary.

---

## Speed

Speed varied significantly across models.

| Model | Average Speed |
|------|------|
| distilbart-cnn-6-6 | ~0.94s |
| flan-t5-small | ~1.55s |
| distilbart-cnn-12-6 | ~3.28s |
| bart-large-cnn-samsum | ~5.75s |
| flan-t5-large | ~11.22s |

Although **distilbart models were faster**, they produced lower-quality summaries. Also, the XL AND XXL models took a few minutes to run which wouldn't be ideal!

For the Echolabs use case, since accuracy is a lot more important, a 5-7 seconds latency can be considered acceptable for now. 

---

## Relevance

Relevance measures whether the summary captures the main ideas from the transcript.

Example transcript topics included:

- A project update where the backend is unfinished
- A product roadmap with three strategic goals

The **bart-large-cnn-samsum** model captured key ideas including:

- Project progress and delays
- Backend completion issues
- Need for a team meeting
- The roadmap goals

Some other models generated:

- News-style sentences unrelated to the input
- Hallucinated stories
- Repetitions of filler phrases

---

## Consistency

Consistency measures whether the model produces similar quality summaries across multiple transcripts.

The **bart-large-cnn-samsum** model showed the highest consistency, producing coherent summaries for both test inputs.

Other models were less reliable:

- **distilbart** sometimes summarized correctly but often repeated parts of the transcript
- **xsum and mnli models** hallucinated unrelated information

---

## Prompting Strategy Findings

Four prompting strategies were tested:

- Zero-shot
- One-shot
- Few-shot
- Many-shot

The results showed that **zero-shot prompting produced the most reliable summaries**.

Adding example summaries in one-shot, few-shot, and many-shot prompts caused models to:

- Copy information from the examples
- Mix example content with the transcript
- Ignore the actual speech input

This suggests that **simpler prompts work better for smaller summarization models**.

---

## Final Model Selection

Based on the evaluation results, the best model for the Echolabs summarization system is:

**philschmid/bart-large-cnn-samsum**

### Reasons for Selection

- Highest overall summary quality
- Strong handling of informal spoken language
- No hallucinations observed during testing
- Consistent results across multiple transcripts
- Acceptable inference speed for near real-time applications

Although some models were faster, they produced significantly lower-quality summaries.

Therefore, **bart-large-cnn-samsum is the most suitable model for integration into the backend.**